# 12.1 工具使用 (Tool Use)

工具使用是Agent的核心能力，使模型能调用外部工具完成自身无法完成的任务。

本节涵盖：
- 函数调用与工具Schema设计
- ReAct推理-行动循环
- 并行工具调用与错误处理
- 工具学习与自适应选择
- 产业级工具调用系统设计

## 1. 函数调用与工具Schema设计

**核心流程**：
1. 定义工具schema（名称、描述、参数JSON Schema）
2. 将工具描述注入系统提示
3. 模型生成结构化函数调用JSON
4. 执行引擎解析并调用对应函数
5. 将结果注入上下文，模型继续生成

**产业实践**：OpenAI Function Calling、Anthropic Tool Use、Google Function Calling均采用类似架构，
核心差异在于schema格式和调用协议。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import math
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field

torch.manual_seed(42)

@dataclass
class ToolParameter:
    name: str
    type: str
    description: str
    required: bool = True
    enum: Optional[List[str]] = None
    default: Any = None

@dataclass
class ToolSchema:
    name: str
    description: str
    parameters: List[ToolParameter]
    returns: str = "str"
    examples: List[Dict] = field(default_factory=list)

    def to_openai_format(self):
        params = {
            "type": "object",
            "properties": {},
            "required": []
        }
        for p in self.parameters:
            prop = {"type": p.type, "description": p.description}
            if p.enum:
                prop["enum"] = p.enum
            if p.default is not None:
                prop["default"] = p.default
            params["properties"][p.name] = prop
            if p.required:
                params["required"].append(p.name)
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": params
            }
        }

weather_tool = ToolSchema(
    name="get_weather",
    description="Get current weather information for a specified location",
    parameters=[
        ToolParameter("location", "string", "City name or coordinates", required=True),
        ToolParameter("unit", "string", "Temperature unit", required=False,
                      enum=["celsius", "fahrenheit"], default="celsius"),
    ],
    examples=[{"location": "Beijing", "unit": "celsius"}]
)

search_tool = ToolSchema(
    name="web_search",
    description="Search the web for information",
    parameters=[
        ToolParameter("query", "string", "Search query string", required=True),
        ToolParameter("num_results", "integer", "Number of results", required=False, default=5),
    ]
)

calc_tool = ToolSchema(
    name="calculator",
    description="Evaluate a mathematical expression safely",
    parameters=[
        ToolParameter("expression", "string", "Math expression to evaluate", required=True),
    ]
)

tools = [weather_tool, search_tool, calc_tool]

print('=== Tool Schema Design (OpenAI Function Calling Format) ===')
for t in tools:
    schema = t.to_openai_format()
    print(f'\nTool: {t.name}')
    print(f'  Description: {t.description}')
    print(f'  Schema: {json.dumps(schema, indent=2)}')

print(f'\nKey: Industry-standard tool schemas use JSON Schema for parameter validation.')
print(f'OpenAI/Anthropic/Google all follow this pattern with minor format differences.')

## 2. 工具调用生成器 — 神经网络实现

模型需要同时完成两个任务：
- **工具选择**：从N个工具中选择最合适的一个（或选择不调用工具）
- **参数生成**：为选中工具生成正确的参数值

**训练方法**：
- 监督学习：在(查询, 工具调用)对上训练
- 强化学习：根据工具执行结果反馈优化选择策略
- 拒绝学习：训练模型识别何时不应调用工具

In [ ]:
class ToolCallGenerator(nn.Module):
    def __init__(self, d_model=128, n_tools=4, d_vocab=256, max_arg_len=32):
        super().__init__()
        self.n_tools = n_tools
        self.d_model = d_model
        self.max_arg_len = max_arg_len

        self.query_encoder = nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=256, batch_first=True)
        self.tool_classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_tools + 1)
        )
        self.confidence_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.arg_decoder = nn.GRU(d_model, 64, num_layers=2, batch_first=True)
        self.arg_output = nn.Linear(64, d_vocab)

    def forward(self, query_embed):
        encoded = self.query_encoder(query_embed.unsqueeze(1)).squeeze(1)
        tool_logits = self.tool_classifier(encoded)
        tool_probs = F.softmax(tool_logits, dim=-1)
        confidence = self.confidence_head(encoded)
        arg_hidden = torch.zeros(2, query_embed.size(0), 64, device=query_embed.device)
        arg_input = encoded.unsqueeze(1).expand(-1, self.max_arg_len, -1)
        arg_out, _ = self.arg_decoder(arg_input, arg_hidden)
        arg_logits = self.arg_output(arg_out)
        return tool_probs, confidence, arg_logits

    def select_tool(self, query_embed, confidence_threshold=0.5):
        tool_probs, confidence, arg_logits = self.forward(query_embed)
        if confidence.item() < confidence_threshold:
            return None, 0.0, None
        selected = tool_probs.argmax(dim=-1)
        if selected.item() >= self.n_tools:
            return None, confidence.item(), None
        return selected.item(), confidence.item(), arg_logits

generator = ToolCallGenerator(d_model=128, n_tools=3, d_vocab=64, max_arg_len=16)
query = torch.randn(1, 128)

tool_probs, confidence, arg_logits = generator(query)
selected_idx, conf, args = generator.select_tool(query, confidence_threshold=0.3)

print('=== Neural Tool Call Generator ===')
print(f'Tool probabilities (incl. no-call): {[f"{p:.3f}" for p in tool_probs[0].tolist()]}')
print(f'Confidence: {confidence.item():.3f}')
print(f'Selected: {selected_idx} (tool: {tools[selected_idx].name if selected_idx is not None else "none"})')
print(f'Arg logits shape: {arg_logits.shape if args is not None else "N/A"}')
print(f'\nKey: The +1 class in tool_classifier represents "no tool call" — crucial for avoiding')
print(f'unnecessary tool invocations. Confidence threshold prevents low-certainty calls.')

## 3. ReAct推理-行动循环

**ReAct** (Reasoning + Acting) 是产业界最广泛使用的Agent框架：
- **Thought**：模型推理当前状态，决定下一步行动
- **Action**：调用工具执行操作
- **Observation**：接收工具返回结果
- 循环直到任务完成

**与纯CoT的区别**：CoT只在模型内部推理，无法获取外部信息；ReAct通过工具调用打破这一限制。

**产业应用**：LangChain ReAct Agent、AutoGPT、BabyAGI等均基于此范式。

In [ ]:
class ReActAgent(nn.Module):
    def __init__(self, d_model=128, n_tools=3, max_steps=6):
        super().__init__()
        self.n_tools = n_tools
        self.max_steps = max_steps

        self.thought_encoder = nn.GRU(d_model, 64, num_layers=2, batch_first=True)
        self.action_head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, n_tools + 1)
        )
        self.finish_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.obs_encoder = nn.Linear(d_model, 64)
        self.answer_head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, d_model)
        )

    def forward(self, query, observations=None):
        seq = [query]
        if observations:
            for obs in observations:
                seq.append(self.obs_encoder(obs))
        input_seq = torch.stack(seq, dim=1)
        gru_out, h = self.thought_encoder(input_seq)
        thought = gru_out[:, -1, :]
        action_logits = self.action_head(thought)
        should_finish = self.finish_head(thought)
        answer = self.answer_head(thought)
        return action_logits, should_finish, answer, thought

class ToolExecutor:
    def __init__(self, tools, d_model=128):
        self.tools = {i: t for i, t in enumerate(tools)}
        self.d_model = d_model
        self.call_history = []

    def execute(self, tool_idx, **kwargs):
        if tool_idx not in self.tools:
            return self._error_result(f'Unknown tool index: {tool_idx}')
        tool = self.tools[tool_idx]
        self.call_history.append({'tool': tool.name, 'args': kwargs})
        return self._mock_result(tool.name, kwargs)

    def _mock_result(self, tool_name, kwargs):
        result = torch.randn(1, self.d_model) * 0.5
        if tool_name == 'get_weather':
            result = result + 1.0
        elif tool_name == 'web_search':
            result = result + 0.5
        elif tool_name == 'calculator':
            result = result - 0.3
        return result

    def _error_result(self, msg):
        return torch.zeros(1, self.d_model)

agent = ReActAgent(d_model=128, n_tools=3, max_steps=6)
executor = ToolExecutor(tools, d_model=128)

query = torch.randn(1, 128)
observations = []

print('=== ReAct Loop: Reasoning + Acting ===')
print(f'Query: [embedded vector]')
for step in range(agent.max_steps):
    action_logits, should_finish, answer, thought = agent(query, observations)
    action_probs = F.softmax(action_logits, dim=-1)
    selected_action = action_probs.argmax(dim=-1).item()
    finish_prob = should_finish.item()

    print(f'\nStep {step+1}:')
    print(f'  Thought: [encoded reasoning state, norm={thought.norm():.3f}]')
    print(f'  Action: {executor.tools.get(selected_action, "FINISH").name if selected_action < 3 else "FINISH"}')
    print(f'  Action confidence: {action_probs.max().item():.3f}')
    print(f'  Finish probability: {finish_prob:.3f}')

    if finish_prob > 0.7 or selected_action >= 3:
        print(f'  -> Agent decides to FINISH with answer')
        break

    obs = executor.execute(selected_action)
    observations.append(obs)
    print(f'  Observation: [tool result, norm={obs.norm():.3f}]')

print(f'\nTotal tool calls: {len(executor.call_history)}')
print(f'Call history: {[h["tool"] for h in executor.call_history]}')
print(f'\nKey: ReAct alternates between reasoning (thought) and tool use (action).')
print(f'The finish gate prevents infinite loops — critical for production systems.')

## 4. 并行工具调用与错误处理

**并行调用**：当多个工具之间无依赖关系时，可同时调用以提高效率。
例如：查询北京天气 + 查询上海天气可并行执行。

**错误处理策略**：
- **重试**：对瞬时错误（网络超时、限流）进行指数退避重试
- **降级**：工具不可用时切换到备选方案
- **回传**：将错误信息作为Observation反馈给Agent，让其调整策略
- **超时**：设置工具执行超时，防止阻塞

**产业实践**：OpenAI支持parallel_tool_calls，LangChain提供ToolException和retry机制。

In [ ]:
import time

class ParallelToolExecutor:
    def __init__(self, tools, d_model=128, max_retries=3, timeout_sec=5.0):
        self.tools = {i: t for i, t in enumerate(tools)}
        self.d_model = d_model
        self.max_retries = max_retries
        self.timeout = timeout_sec
        self.call_stats = {'total': 0, 'success': 0, 'retries': 0, 'failures': 0}

    def execute_single(self, tool_idx, **kwargs):
        for attempt in range(self.max_retries):
            try:
                self.call_stats['total'] += 1
                if torch.rand(1).item() < 0.15 and attempt < self.max_retries - 1:
                    raise ConnectionError(f'Tool {self.tools[tool_idx].name} temporarily unavailable')
                result = self._mock_execute(tool_idx, kwargs)
                self.call_stats['success'] += 1
                return result, None
            except Exception as e:
                self.call_stats['retries'] += 1
                if attempt == self.max_retries - 1:
                    self.call_stats['failures'] += 1
                    return self._error_result(str(e)), str(e)
                backoff = (2 ** attempt) * 0.1
                time.sleep(backoff * 0.01)
        return self._error_result('Max retries exceeded'), 'Max retries exceeded'

    def execute_parallel(self, tool_calls):
        results = []
        for tool_idx, kwargs in tool_calls:
            result, error = self.execute_single(tool_idx, **kwargs)
            results.append({'result': result, 'error': error, 'tool': self.tools[tool_idx].name})
        return results

    def _mock_execute(self, tool_idx, kwargs):
        return torch.randn(1, self.d_model) * 0.5 + tool_idx

    def _error_result(self, msg):
        return torch.zeros(1, self.d_model)

class ParallelCallPlanner(nn.Module):
    def __init__(self, d_model=128, n_tools=3):
        super().__init__()
        self.dependency_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Linear(64, n_tools * n_tools)
        )
        self.call_set_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Linear(64, n_tools), nn.Sigmoid()
        )
        self.n_tools = n_tools

    def forward(self, query):
        dep_matrix = self.dependency_net(query).view(self.n_tools, self.n_tools)
        dep_matrix = (dep_matrix > 0).float()
        call_mask = self.call_set_net(query)
        call_mask = (call_mask > 0.5).float()
        parallel_groups = self._find_parallel_groups(dep_matrix, call_mask)
        return parallel_groups, dep_matrix, call_mask

    def _find_parallel_groups(self, dep_matrix, call_mask):
        called = [i for i in range(self.n_tools) if call_mask[0, i] > 0.5]
        groups = []
        remaining = set(called)
        completed = set()
        while remaining:
            ready = []
            for t in remaining:
                deps = set(i for i in range(self.n_tools) if dep_matrix[t, i] > 0.5)
                if deps.issubset(completed):
                    ready.append(t)
            if not ready:
                ready = [min(remaining)]
            groups.append(ready)
            for t in ready:
                remaining.discard(t)
                completed.add(t)
        return groups

planner = ParallelCallPlanner(d_model=128, n_tools=3)
executor = ParallelToolExecutor(tools, d_model=128)

query = torch.randn(1, 128)
parallel_groups, dep_matrix, call_mask = planner(query)

print('=== Parallel Tool Calling ===')
print(f'Call mask (which tools to call): {call_mask[0].int().tolist()}')
print(f'Dependency matrix:')
for i, row in enumerate(dep_matrix):
    print(f'  Tool {i} depends on: {row.int().tolist()}')
print(f'\nExecution groups (parallel within each group):')
for g_idx, group in enumerate(parallel_groups):
    tool_names = [tools[i].name for i in group]
    print(f'  Group {g_idx}: {tool_names}')
    results = executor.execute_parallel([(i, {}) for i in group])
    for r in results:
        status = 'OK' if r['error'] is None else f'ERROR: {r["error"]}'
        print(f'    {r["tool"]}: {status}')

print(f'\nExecutor stats: {executor.call_stats}')
print(f'\nKey: Dependency analysis enables safe parallel execution.')
print(f'Exponential backoff retry handles transient failures gracefully.')

## 5. 工具学习与自适应选择

**核心问题**：如何让模型学会"何时用工具"和"用哪个工具"？

**训练范式**：
1. **SFT阶段**：在人工标注的(查询, 工具调用)对上做监督微调
2. **RL阶段**：使用工具执行结果作为奖励信号，优化工具选择策略
3. **拒绝学习**：加入负样本，训练模型识别不需要工具的查询

**Gorilla方法**：在LLaMA上微调，使用AST匹配评估工具调用的正确性，而非字符串匹配。

In [ ]:
class ToolLearningAgent(nn.Module):
    def __init__(self, d_model=128, n_tools=3):
        super().__init__()
        self.n_tools = n_tools
        self.query_encoder = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.tool_scorer = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
        self.tool_embeddings = nn.Parameter(torch.randn(n_tools, d_model) * 0.1)
        self.no_tool_embed = nn.Parameter(torch.randn(1, d_model) * 0.1)
        self.should_call = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, query):
        q = self.query_encoder(query)
        call_prob = self.should_call(q)
        all_embeds = torch.cat([self.tool_embeddings, self.no_tool_embed], dim=0)
        q_expanded = q.expand(all_embeds.size(0), -1)
        pairs = torch.cat([q_expanded, all_embeds], dim=-1)
        scores = self.tool_scorer(pairs).squeeze(-1)
        tool_probs = F.softmax(scores, dim=0)
        return tool_probs, call_prob

    def compute_loss(self, query, target_tool_idx, should_call_label):
        tool_probs, call_prob = self.forward(query)
        tool_loss = F.cross_entropy(tool_probs.unsqueeze(0),
                                     torch.tensor([target_tool_idx]))
        call_loss = F.binary_cross_entropy(call_prob.squeeze(),
                                            torch.tensor([should_call_label], dtype=torch.float))
        return tool_loss + 0.5 * call_loss, tool_loss, call_loss

agent = ToolLearningAgent(d_model=128, n_tools=3)
optimizer = torch.optim.AdamW(agent.parameters(), lr=1e-3)

training_data = [
    (torch.randn(1, 128), 0, 1.0),
    (torch.randn(1, 128), 1, 1.0),
    (torch.randn(1, 128), 2, 1.0),
    (torch.randn(1, 128), 3, 0.0),
    (torch.randn(1, 128), 3, 0.0),
]

print('=== Tool Learning Training ===')
for epoch in range(20):
    total_loss = 0
    for query, target, call_label in training_data:
        loss, t_loss, c_loss = agent.compute_loss(query, target, call_label)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(agent.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}: total_loss={total_loss:.4f}')

print(f'\nInference after training:')
test_queries = [
    ('Weather query', torch.randn(1, 128)),
    ('Math query', torch.randn(1, 128)),
    ('General chat', torch.randn(1, 128)),
]
for name, q in test_queries:
    probs, call_p = agent(q)
    tool_names = [t.name for t in tools] + ['no_tool']
    best = probs.argmax().item()
    print(f'  {name}: best={tool_names[best]} (p={probs[best]:.3f}), call_prob={call_p.item():.3f}')

print(f'\nKey: Tool learning combines tool selection with a should-call gate.')
print(f'The no_tool embedding teaches the model when NOT to use tools.')

## 6. 产业级工具调用系统架构

生产环境中的工具调用系统需要考虑：

### 可靠性
- **重试与超时**：指数退避 + 抖动，防止雪崩
- **熔断器**：连续失败时暂停调用，避免级联故障
- **降级策略**：主工具不可用时切换备选

### 安全性
- **权限控制**：不同角色可访问不同工具
- **输入验证**：严格校验工具参数，防止注入攻击
- **审计日志**：记录所有工具调用，便于追溯

### 可观测性
- **调用追踪**：分布式追踪（如OpenTelemetry）
- **指标监控**：延迟、成功率、工具使用分布
- **成本控制**：API调用计费与预算限制

In [ ]:
class CircuitBreaker:
    CLOSED = 'closed'
    OPEN = 'open'
    HALF_OPEN = 'half_open'

    def __init__(self, failure_threshold=3, recovery_timeout=5.0, half_open_max=1):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.half_open_max = half_open_max
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0
        self.half_open_calls = 0

    def can_execute(self):
        if self.state == self.CLOSED:
            return True
        if self.state == self.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = self.HALF_OPEN
                self.half_open_calls = 0
                return True
            return False
        if self.state == self.HALF_OPEN:
            return self.half_open_calls < self.half_open_max
        return False

    def record_success(self):
        if self.state == self.HALF_OPEN:
            self.state = self.CLOSED
        self.failure_count = 0

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.state == self.HALF_OPEN:
            self.state = self.OPEN
        elif self.failure_count >= self.failure_threshold:
            self.state = self.OPEN

class ProductionToolExecutor:
    def __init__(self, tools, d_model=128, max_retries=3, budget_limit=100):
        self.tools = {i: t for i, t in enumerate(tools)}
        self.d_model = d_model
        self.max_retries = max_retries
        self.budget_limit = budget_limit
        self.budget_used = 0
        self.circuit_breakers = {i: CircuitBreaker() for i in range(len(tools))}
        self.audit_log = []
        self.metrics = {'calls': 0, 'successes': 0, 'latencies': [], 'errors': defaultdict(int)}

    def execute(self, tool_idx, user_role='user', **kwargs):
        tool = self.tools[tool_idx]
        self.audit_log.append({'tool': tool.name, 'role': user_role, 'args': kwargs, 'time': time.time()})

        if self.budget_used >= self.budget_limit:
            return None, 'Budget exceeded'

        cb = self.circuit_breakers[tool_idx]
        if not cb.can_execute():
            return None, f'Circuit breaker OPEN for {tool.name}'

        start = time.time()
        for attempt in range(self.max_retries):
            try:
                self.metrics['calls'] += 1
                result = self._mock_execute(tool_idx, kwargs)
                latency = time.time() - start
                self.metrics['successes'] += 1
                self.metrics['latencies'].append(latency)
                cb.record_success()
                self.budget_used += 1
                return result, None
            except Exception as e:
                self.metrics['errors'][str(e)] += 1
                if attempt == self.max_retries - 1:
                    cb.record_failure()
                    return None, str(e)
                time.sleep((2 ** attempt) * 0.01)
        return None, 'Max retries exceeded'

    def _mock_execute(self, tool_idx, kwargs):
        return torch.randn(1, self.d_model)

    def get_metrics(self):
        avg_latency = sum(self.metrics['latencies']) / max(len(self.metrics['latencies']), 1)
        success_rate = self.metrics['successes'] / max(self.metrics['calls'], 1)
        return {'avg_latency': avg_latency, 'success_rate': success_rate,
                'total_calls': self.metrics['calls'], 'budget_used': self.budget_used}

prod_executor = ProductionToolExecutor(tools, budget_limit=10)

print('=== Production Tool Executor ===')
for i in range(5):
    result, error = prod_executor.execute(i % 3, user_role='admin')
    status = 'OK' if error is None else f'ERROR: {error}'
    print(f'Call {i+1}: tool={tools[i%3].name}, status={status}')

print(f'\nMetrics: {prod_executor.get_metrics()}')
print(f'Circuit breaker states: {[(tools[i].name, cb.state) for i, cb in prod_executor.circuit_breakers.items()]}')
print(f'Audit log entries: {len(prod_executor.audit_log)}')
print(f'\nKey: Production systems need circuit breakers, budgets, audit logs, and metrics.')
print(f'These prevent cascading failures and enable observability.')

## 7. 工具调用总结与最佳实践

| 维度 | 关键要点 |
|------|----------|
| Schema设计 | 使用JSON Schema，包含描述、枚举、默认值 |
| 工具选择 | 双头架构：选择头+置信度头，支持拒绝调用 |
| 推理框架 | ReAct循环：Thought→Action→Observation |
| 并行调用 | 依赖分析→拓扑排序→分组并行执行 |
| 错误处理 | 重试+熔断+降级+超时四重保障 |
| 安全 | 权限控制+输入验证+审计日志 |
| 训练 | SFT→RL两阶段，加入拒绝学习负样本 |

**前沿方向**：
- **Toolformer**：模型自主学习使用工具
- **Gorilla**：大规模API调用微调
- **ToolLLM**：构建大规模工具学习基准
- **ActFromTag**：从HTML标签学习工具使用